# Tangerine Tutorial
Install and run the `tangerine` pipeline on the intermediate neural progenitors dataset.

## Installation instructions
Clone the repository 
```bash
git clone https://github.com/ntanmayee/tangerine.git
cd tangerine
```

Install `tangerine` with poetry by running these commands

```bash
conda config --add channels bioconda
conda config --add channels conda-forge
conda create -n tang python=3.10 scanpy python-igraph leidenalg typer poetry pysam=0.22 bioconda::htseq conda-forge::pyarrow

conda activate tang

pip install dash gimmemotifs --no-cache-dir
conda install dash-bootstrap-components
pip install dash-cytoscape==0.2.0
poetry install --only-root
```

Install the genome of your choice. Here we'll be using mouse data, so let's install the `mm10` genome.
```bash
genomepy install mm10 --provider UCSC --annotation
```

You will also need a BED file with the list of genes and their sequence information. It should have `chr`, `start`, `end` and `gene_name` columns.

## Download the data
We'll be using the intermediate neural progenitors subset of cells from a large dataset of mouse development. You can read more about the dataset [here](https://cellxgene.cziscience.com/collections/45d5d2c3-bc28-4814-aed6-0bb6f0e11c82). 

The dataset is about 7 GB, so make sure you have enough disk space before running this command.
```bash
wget https://datasets.cellxgene.cziscience.com/e777de2e-aa5f-4907-b4ad-a8bee136d4ae.h5ad
```

## Preprocess data
Before we run `tangerine`, we need to do some basic preprocessing.

In [1]:
import scanpy as sc

In [2]:
adata = sc.read_h5ad('intermediate_neural_progenitors.h5ad')
adata

AnnData object with n_obs × n_vars = 628251 × 45525
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'author_experimental_id', 'author_day', 'author_somite_count', 'author_major_cell_cluster', 'author_cell_type', 'tissue_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'gene_short_name', 'chr', 'start', 'end', 'strand', 'gene_type', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'author_day_colors', 'author_somite_count_colors', 'citation', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_umap'
    layers: 'raw.X'

In [3]:
# clean gene names
adata.var['ensembl_id'] = adata.var.index
adata.var.index = adata.var['gene_short_name'].astype(str)
adata.var_names_make_unique()

adata.var.drop(columns=['gene_short_name'], inplace=True)
adata.var.index.name = None

adata.var.head()

,chr,start,end,strand,gene_type,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type,ensembl_id
4933401J01Rik,chr1,3073253,3074322,+,TEC,False,4933401J01Rik,NCBITaxon:10090,gene,1070,TEC,ENSMUSG00000102693
Gm26206,chr1,3102016,3102125,+,snRNA,False,Gm26206,NCBITaxon:10090,gene,110,snRNA,ENSMUSG00000064842
Xkr4,chr1,3205901,3671498,-,protein_coding,False,Xkr4,NCBITaxon:10090,gene,3634,protein_coding,ENSMUSG00000051951
Gm18956,chr1,3252757,3253236,+,processed_pseudogene,False,Gm18956,NCBITaxon:10090,gene,480,processed_pseudogene,ENSMUSG00000102851
Gm37180,chr1,3365731,3368549,-,TEC,False,Gm37180,NCBITaxon:10090,gene,2819,TEC,ENSMUSG00000103377


In [4]:
# find cells per timepoint
stats = adata.obs.groupby('author_day').size()
stats

/tmp/3815766.1.rhel9.q/ipykernel_1593041/3750441111.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = adata.obs.groupby('author_day').size()


author_day
E0975           3
E1000           3
E1025          93
E1050          38
E1075         322
E1100         314
E1125         857
E1150        1355
E1175        1226
E1200        3807
E1225        6842
E1250        5650
E1275        2281
E1300        7251
E1325        8871
E1350        3797
E1375        8215
E1400       23583
E1425       11101
E1433.3     12975
E1475        7571
E1500       14188
E1525       12002
E1550       15443
E1575       19966
E1600       13477
E1625        9360
E1650       17978
E1675       14034
E1700       13664
E1725       10389
E1750       11798
E1775       19366
E1800       13126
E1825       18282
E1850       26804
E1875       13919
P0000      278300
dtype: int64

In [5]:
cell_threshold = 10000
stats[stats > cell_threshold].index

CategoricalIndex(['E1400', 'E1425', 'E1433.3', 'E1500', 'E1525', 'E1550',
                  'E1575', 'E1600', 'E1650', 'E1675', 'E1700', 'E1725',
                  'E1750', 'E1775', 'E1800', 'E1825', 'E1850', 'E1875',
                  'P0000'],
                 categories=['E0975', 'E1000', 'E1025', 'E1050', ..., 'E1825', 'E1850', 'E1875', 'P0000'], ordered=False, dtype='category', name='author_day')

In [6]:
# remove postnatal day as it has too many cells
keep_times = ['E1400', 'E1425', 'E1433.3', 'E1500', 'E1525', 'E1550',
              'E1575', 'E1600', 'E1650', 'E1675', 'E1700', 'E1725',
              'E1750', 'E1775', 'E1800', 'E1825', 'E1850', 'E1875']
adata_subset = adata[adata.obs['author_day'].isin(keep_times)].copy()
adata_subset

AnnData object with n_obs × n_vars = 282095 × 45525
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'author_experimental_id', 'author_day', 'author_somite_count', 'author_major_cell_cluster', 'author_cell_type', 'tissue_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'chr', 'start', 'end', 'strand', 'gene_type', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'ensembl_id'
    uns: 'author_day_colors', 'author_somite_count_colors', 'citation', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_umap'
    layers: 'raw.X'

In [ ]:
# save to file
adata_subset.write("intermediate_neural_progenitors_10k_cells.h5ad")

## Run `tangerine` preprocessing
Now let's run the preprocessing pipeline on the data.

```bash
tangerine process -ap intermediate_neural_progenitors_10k_cells.h5ad \
                -g mm10 \
                -t author_day \
                -tp "E1400 E1425 E1433.3 E1500 E1525 E1550 E1575 E1600 E1650 E1675 E1700 E1725 E1750 E1775 E1800 E1825 E1850 E1875" \
                -sw 10000 \
                -sp embryogenesis_neural_10k_cells \
                -b mm10_genes.bed \
                -m kmeans \
                -dt 50
```

This will save all the results to `embryogenesis_neural_10k_cells`.

## Run `tangerine` visualisation
Start the `Plotly dash` app with this command -

```bash
tangerine visualise -tp "E1400 E1425 E1433.3 E1500 E1525 E1550 E1575 E1600 E1650 E1675 E1700 E1725 E1750 E1775 E1800 E1825 E1850 E1875" \ 
        -sp embryogenesis_neural_10k_cells
```

Now open http://localhost:8050 to see the visualisation.